In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Определение модели LeNet
class LeNet(nn.Module):
    def __init__(self, num_channels=1, num_classes=10):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(num_channels, 20, kernel_size=5)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(20, 50, kernel_size=5)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(50 * 4 * 4, 500)   # после двух свёрток и пулинга для 28x28 -> 4x4
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(500, num_classes)
        self.log_softmax = nn.LogSoftmax(dim=1)
    
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1)   # flatten
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return self.log_softmax(x)

# Параметры
batch_size = 64
epochs = 5
learning_rate = 1e-3

# Загрузка данных KMNIST (встроен в torchvision)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.KMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.KMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Модель, функция потерь, оптимизатор
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LeNet(num_channels=1, num_classes=10).to(device)
criterion = nn.NLLLoss()   # negative log likelihood (совместим с LogSoftmax)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Обучение
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Оценка на тесте
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            output = model(images)
            pred = output.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    print(f"PyTorch LeNet - Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Test Acc: {acc:.4f}")